In [255]:
import geopandas as gpd
import pandas as pd
import shapely
import copy
import os
python_directory = os.getcwd()

In [256]:
paths = gpd.read_file(rf"{python_directory}/paths.geojson")
greens = gpd.read_file(rf"{python_directory}/green.geojson")
entry_points = gpd.read_file(rf"{python_directory}/entry_points.geojson")

# Extract nodes and links: Paths
## Example
An example for extracting the nodes:
1. Extract the x and y values from the paths geometry
2. Loop through the extracted x and y values and append the x, y, xy tuples and ids to new lists. The xy tupes are necessary to check if any points are repeated. This can occur when checking other multistrings in case of node intersections
3. Generate a pandas DataFrame listing the nodes and their coordinates

In [272]:
path = paths["geometry"][0]
print(paths["geometry"][0])

# Extract x and y coordinates of the nodes in each line in sequence
x,y = path.geoms[0].xy
x = [round(e, 5) for e in x]
y = [round(e, 5) for e in y]
xy = list(zip(x,y))
print(xy)

# Extract x and y coordinates of the entry points
xy_entry = []
for point in entry_points["geometry"]:
    x_temp, y_temp = point.xy
    xy_entry.append((x_temp[0], y_temp[0]))
print(xy_entry)

# Initialise the lists to captrure the nodes and paths properties
# nodes
xy_list = []
x_list = []
y_list = []
n_id_list = []
n_entry_list = []
# links
l_id_list = []
l_start_list = []
l_end_list = []
max_speed_list = []

n_id_counter = 0
l_id_counter = 0
# Iterate through the nodes indices
for i in range(len(x)):
    #-----------------
    # Nodes
    #-----------------
    # Check if the node has already been given an id
    if (x[i], y[i]) not in xy_list:
        # Identify entry points
        if (x[i], y[i]) in xy_entry: n_entry_list.append(1)
        else: n_entry_list.append(-1)
        # Append relevant parameters
        xy_list.append((x[i], y[i]))
        x_list.append(x[i])
        y_list.append(y[i])
        n_id_list.append(f"n_id{n_id_counter}")
        n_id_counter += 1
    #-----------------
    # Links
    #-----------------
    if i > 0:
        # Find the indices of the nodes in the full xy_lists (including previous geometries)
        i_start = xy_list.index((x[i - 1], y[i - 1]))
        i_end = xy_list.index((x[i], y[i]))
        if i_start == i_end: continue
        # Append the links
        l_start_list.append(n_id_list[i_start])
        l_end_list.append(n_id_list[i_end])
        l_id_list.append(f"l_id{l_id_counter}")
        max_speed_list.append(0)
        l_id_counter += 1

df_dict_nodes = {
        "id": n_id_list,
        "xpos": x_list,
        "ypos": y_list,
        "entry": n_entry_list
    }
df_dict_links = {
    "id": l_id_list,
    "start": l_start_list,
    "end": l_end_list
}
df_nodes = pd.DataFrame(df_dict_nodes)
df_links = pd.DataFrame(df_dict_links)
display(df_nodes)
display(df_links)


MULTILINESTRING ((449978.8425175871 207967.26966033806, 449978.67416675494 207996.43991002906, 449977.59470474714 208009.9331851264, 449978.67416675494 208014.79076416144, 449981.3728217744 208021.80726721205, 449982.9920147861 208033.1416182938, 449984.6112077978 208048.2540864028, 449981.9125527783 208061.74736150014, 449979.75362876273 208067.14467153908, 449975.4357807316 208070.92278856633, 449970.57820169657 208074.16117458968, 449965.1808916576 208076.32009860524, 449940.89299648243 208077.93929161693, 449932.25730042014 208080.6379466364, 449930.09837640455 208081.71740864418, 449921.46268034226 208090.89283571037, 449920.3832183345 208093.59149072983, 449920.3832183345 208102.76691779602, 449921.46268034226 208117.87938590502, 449925.78052837343 208130.83292999843, 449926.8599903812 208135.69050903348, 449931.17783841235 208143.78647409187, 449933.33676242793 208151.88243915027, 449931.17783841235 208166.99490725927, 449927.3997213851 208173.471679306, 449926.8599903812 208176

,id,xpos,ypos,entry
0,n_id0,449978.84252,207967.26966,-1
1,n_id1,449978.67417,207996.43991,-1
2,n_id2,449977.59470,208009.93319,-1
3,n_id3,449978.67417,208014.79076,-1
4,n_id4,449981.37282,208021.80727,-1
...,...,...,...,...
72,n_id72,449554.44560,208634.40196,-1
73,n_id73,449547.42909,208636.02115,-1
74,n_id74,449539.87286,208638.71980,-1
75,n_id75,449536.63447,208640.33900,-1


,id,start,end
0,l_id0,n_id0,n_id1
1,l_id1,n_id1,n_id2
2,l_id2,n_id2,n_id3
3,l_id3,n_id3,n_id4
4,l_id4,n_id4,n_id5
...,...,...,...
71,l_id71,n_id71,n_id72
72,l_id72,n_id72,n_id73
73,l_id73,n_id73,n_id74
74,l_id74,n_id74,n_id75


In [ ]:
def extract_nodes_links(paths, entry_points) -> pd.DataFrame:
    """
    Extracts the nodes from a geopandas Multistring dataframe.

    Parameters
    ----------
    - `paths`: geopandas DataFrame
        The geopandas dataframe including lines representing paths inside a green space

    Returns
    -------
    - geopandas Dataframe including the node points and their ids
    """
    # Extract x and y coordinates of the entry points
    xy_entry = []
    for point in entry_points["geometry"]:
        x_temp, y_temp = point.xy
        x_temp = [round(e, 5) for e in x_temp]
        y_temp = [round(e, 5) for e in y_temp]
        xy_entry.append((x_temp[0], y_temp[0]))
    print(xy_entry)

    # Initialise the lists to captrure the nodes and paths properties
    xy_list = []
    x_list = []
    y_list = []
    n_id_list = []
    n_sp_list = []
    n_entry_list = []
    l_id_list = []
    l_start_list = []
    l_end_list = []
    max_speed_list = []
    
    n_id_counter = 0
    l_id_counter = 0
    for path in paths["geometry"]:
        x,y = path.geoms[0].xy
        x = [round(e, 5) for e in x]
        y = [round(e, 5) for e in y]
        # Iterate through the nodes indices
        for i in range(len(x)):
            #-----------------
            # Nodes
            #-----------------
            # Check if the node has already been given an id
            if (x[i], y[i]) not in xy_list:
                xy_list.append((x[i], y[i]))
                x_list.append(x[i])
                y_list.append(y[i])
                n_id_list.append(f"n_id{n_id_counter}")
                n_sp_list.append(-1)
                n_id_counter += 1
                # Identify entry points
                if (x[i], y[i]) in xy_entry: n_entry_list.append(1)
                else: n_entry_list.append(-1)
            #-----------------
            # Links
            #-----------------
            if i > 0:
                # Find the indices of the nodes in the full xy_lists (including previous geometries)
                i_start = xy_list.index((x[i - 1], y[i - 1]))
                i_end = xy_list.index((x[i], y[i]))
                if i_start == i_end: continue
                # Append the links
                l_start_list.append(n_id_list[i_start])
                l_end_list.append(n_id_list[i_end])
                l_id_list.append(f"l_id{l_id_counter}")
                max_speed_list.append(0)
                l_id_counter += 1

    df_dict_nodes = {
        "id": n_id_list,
        "xpos": x_list,
        "ypos": y_list,
        "sp_id": n_sp_list,
        "entry": n_entry_list
    }
    df_dict_links = {
        "id": l_id_list,
        "start": l_start_list,
        "end": l_end_list,
        "maxspeed": max_speed_list
    }
    df_nodes = pd.DataFrame(df_dict_nodes)
    df_links = pd.DataFrame(df_dict_links)
    return df_nodes, df_links

## Application

In [270]:
gdf_paths = gpd.read_file(rf"{python_directory}/paths.geojson")
gdf_entry_points = gpd.read_file(rf"{python_directory}/entry_points.geojson")

df_nodes, df_links = extract_nodes_links(gdf_paths, gdf_entry_points)

display(df_nodes)
display(df_links)
display(df_nodes.iloc[[109]])
print(xy_entry)

,id,xpos,ypos,sp_id,entry
0,n_id0,449978.842518,207967.269660,-1,1
1,n_id1,449978.674167,207996.439910,-1,-1
2,n_id2,449977.594705,208009.933185,-1,-1
3,n_id3,449978.674167,208014.790764,-1,-1
4,n_id4,449981.372822,208021.807267,-1,-1
...,...,...,...,...,...
306,n_id306,449778.973695,208774.563352,-1,-1
307,n_id307,449790.847777,208771.864697,-1,-1
308,n_id308,449799.483473,208767.546849,-1,-1
309,n_id309,449804.341052,208763.229001,-1,-1


,id,start,end,maxspeed
0,l_id0,n_id0,n_id1,0
1,l_id1,n_id1,n_id2,0
2,l_id2,n_id2,n_id3,0
3,l_id3,n_id3,n_id4,0
4,l_id4,n_id4,n_id5,0
...,...,...,...,...
313,l_id313,n_id306,n_id307,0
314,l_id314,n_id307,n_id308,0
315,l_id315,n_id308,n_id309,0
316,l_id316,n_id309,n_id310,0


,id,xpos,ypos,sp_id,entry
109,n_id109,449628.200635,208958.247141,-1,-1


[(449518.03639987664, 208642.98908741673), (449628.2006345574, 208958.24714116007), (449978.8425175871, 207967.26966033806)]


---
# Extract nodes: Polygons

In [260]:
greens = gpd.read_file(rf"{python_directory}/green_27700.geojson")
df_nodes_paths = df_nodes

In [261]:
green = greens["geometry"][0]
print(greens["geometry"][0])
print(green.exterior.xy)

# Extract x and y coordinates of the nodes in the outer polygon line
x,y = green.exterior.xy
xy = list(zip(x,y))
print(xy)

# Initialise the needed parameters
xy_list = []
x_list = []
y_list = []
n_id_list = []

n_id_counter = len(df_nodes_paths)
for i in range(len(x)):
    #-----------------
    # Nodes
    #-----------------
    # Check if the node has already been given an id
    if (x[i], y[i]) not in xy_list:
        xy_list.append((x[i], y[i]))
        x_list.append(x[i])
        y_list.append(y[i])
        n_id_list.append(f"n_id{n_id_counter}")
        n_id_counter += 1

df_dict_nodes_green = {
        "id": n_id_list,
        "xpos": x_list,
        "ypos": y_list
    }
df_nodes_green = pd.DataFrame(df_dict_nodes_green)
display(df_nodes_green.head())

POLYGON ((449525.363433402 208650.8136254467, 449533.258197289 208655.62945526943, 449639.92041842133 208755.19279469905, 449628.20063455746 208958.24714116007, 449623.33861311746 209026.04858271038, 449807.3072094354 209087.14622262167, 449820.53595959244 209065.9543598994, 449893.353196776 208939.78459366184, 449923.9802412502 208882.01520537143, 449943.7825716318 208841.6128149663, 449970.25013705983 208782.44553302694, 450000.6053771518 208710.51438526675, 450016.3909995207 208673.10893088678, 450030.1388398574 208634.55990994017, 450057.7759222563 208553.70385502337, 450085.14859672287 208460.64354672836, 450110.76231315534 208361.3593634258, 450121.065358268 208321.01896979526, 450118.0772812533 208316.8627092528, 450174.3653695065 208072.61906647595, 450149.68294218543 208061.28422498266, 450054.64463450754 208013.81374194007, 449999.29254710744 207985.85830702662, 449978.8425175871 207967.26966033806, 449974.3840077407 207968.70484555798, 449964.1583363306 207975.49975993345, 4

,id,xpos,ypos
0,n_id311,449525.363433,208650.813625
1,n_id312,449533.258197,208655.629455
2,n_id313,449639.920418,208755.192795
3,n_id314,449628.200635,208958.247141
4,n_id315,449623.338613,209026.048583


In [262]:
df_nodes_collated = pd.concat([df_nodes, df_nodes_green], ignore_index=True)
display(df_nodes_collated)

,id,xpos,ypos,sp_id,entry
0,n_id0,449978.842518,207967.269660,-1.0,1.0
1,n_id1,449978.674167,207996.439910,-1.0,-1.0
2,n_id2,449977.594705,208009.933185,-1.0,-1.0
3,n_id3,449978.674167,208014.790764,-1.0,-1.0
4,n_id4,449981.372822,208021.807267,-1.0,-1.0
...,...,...,...,...,...
345,n_id345,449684.687452,208479.174887,NaN,NaN
346,n_id346,449613.005920,208541.669619,NaN,NaN
347,n_id347,449559.829692,208588.997549,NaN,NaN
348,n_id348,449519.241436,208633.401894,NaN,NaN


In [265]:
df_collated = df_nodes_collated
x_max = max(df_collated["xpos"])
x_min = min(df_collated["xpos"])
y_max = max(df_collated["ypos"])
y_min = min(df_collated["ypos"])

df_nodes_normalised = copy.deepcopy(df_nodes)
df_nodes_normalised["xpos"] = [(xpos - x_min) / (x_max - x_min) for xpos in df_nodes["xpos"]]
df_nodes_normalised["ypos"] = [(ypos - y_min) / (y_max - y_min) for ypos in df_nodes["ypos"]]
display(df_nodes_normalised)
display(df_nodes_normalised[df_nodes_normalised["entry"] == 1])
print(max(df_nodes_normalised["xpos"]))

,id,xpos,ypos,sp_id,entry
0,n_id0,0.702096,0.000000,-1,1
1,n_id1,0.701840,0.026048,-1,-1
2,n_id2,0.700195,0.038097,-1,-1
3,n_id3,0.701840,0.042434,-1,-1
4,n_id4,0.705952,0.048700,-1,-1
...,...,...,...,...,...
306,n_id306,0.397571,0.720877,-1,-1
307,n_id307,0.415663,0.718468,-1,-1
308,n_id308,0.428820,0.714612,-1,-1
309,n_id309,0.436221,0.710756,-1,-1


,id,xpos,ypos,sp_id,entry
0,n_id0,0.702096,0.000000,-1,1
76,n_id76,0.000000,0.603387,-1,1


0.9532727492916534


# Save CSVs

In [264]:
df_nodes_normalised.to_csv(f"{python_directory}/node_list.csv", index=False)
df_links.to_csv(f"{python_directory}/link_list.csv", index=False)

df = df_links
display(df[df["start"] == df["end"]])

display(df_nodes_normalised[df_nodes_normalised["id"] == "n_id258"])


,id,start,end,maxspeed


,id,xpos,ypos,sp_id,entry
258,n_id258,0.472199,0.703286,-1,-1
